# Excel context checks
Step 1 only: structure, coverage, missing values, duplicate keys and metric definition checks.
Source: `data/Social_Media_Content_Performance_Dataset.xlsx`, sheet `Sheet1`.
Requires Python, pandas and openpyxl. No performance rankings, causal analysis, recommendations or models.
See `01_data_context.md` and `../Measurement Context/03_measurement_context.md` for interpretation.


In [1]:
from pathlib import Path
import pandas as pd

# Locate the project from the repository root or a nested notebook folder.
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data/Social_Media_Content_Performance_Dataset.xlsx").exists())
file_path = project_root / "data/Social_Media_Content_Performance_Dataset.xlsx"
workbook = pd.ExcelFile(file_path)
df = pd.read_excel(workbook, sheet_name="Sheet1")
print("Source:", file_path.name)
print("Sheets:", workbook.sheet_names)
print("Rows, columns:", df.shape)
print(df.dtypes.to_string())

Source: Social_Media_Content_Performance_Dataset.xlsx
Sheets: ['Sheet1']
Rows, columns: (5600, 24)
Post_ID                       object
Platform                      object
Content_Type                  object
Content_Category              object
Post_Type                     object
Region                        object
Longitude                    float64
Latitude                     float64
Engagement                     int64
Views                          int64
Likes                          int64
Shares                         int64
Comments                       int64
Engagement_Rate              float64
Impressions                    int64
Video_Views                    int64
Live_Stream_Views              int64
Clicks                       float64
Click_Through_Rate           float64
Main_Hashtag                  object
Post_Published_At     datetime64[ns]
Post_Date             datetime64[ns]
Post_Hour                      int64
Engagement_Level              object


In [2]:
print("Unique Post_ID:", df["Post_ID"].nunique())
print("Repeated Post_ID beyond first:", df.duplicated(["Post_ID"]).sum())
print("Exact duplicate rows beyond first:", df.duplicated().sum())
print("Repeated candidate key beyond first:",
      df.duplicated(["Post_ID", "Platform", "Post_Published_At"]).sum())
print("Missing values:")
print(pd.DataFrame({"missing_count": df.isna().sum(),
                    "missing_pct": df.isna().mean().mul(100).round(2)}).to_string())
print("Clicks / CTR missing masks match:",
      df["Clicks"].isna().equals(df["Click_Through_Rate"].isna()))

Unique Post_ID: 5000
Repeated Post_ID beyond first: 600
Exact duplicate rows beyond first: 0
Repeated candidate key beyond first: 21
Missing values:
                    missing_count  missing_pct
Post_ID                         0         0.00
Platform                        0         0.00
Content_Type                    0         0.00
Content_Category                0         0.00
Post_Type                       0         0.00
Region                          0         0.00
Longitude                       0         0.00
Latitude                        0         0.00
Engagement                      0         0.00
Views                           0         0.00
Likes                           0         0.00
Shares                          0         0.00
Comments                        0         0.00
Engagement_Rate                 0         0.00
Impressions                     0         0.00
Video_Views                     0         0.00
Live_Stream_Views               0         0.00
Click

In [3]:
for column in ["Post_Published_At", "Post_Date", "Post_Hour"]:
    print(column, "min:", df[column].min(), "max:", df[column].max())
for column in ["Platform", "Region", "Content_Category", "Post_Type",
               "Content_Type", "Engagement_Level"]:
    print(column, sorted(df[column].unique()))
print("Click data coverage (not performance):")
print(df.groupby("Platform")["Clicks"].agg(records="size", non_null="count").to_string())

Post_Published_At min: 2024-01-01 10:18:00 max: 2025-05-01 17:08:00
Post_Date min: 2024-01-01 00:00:00 max: 2025-05-01 00:00:00
Post_Hour min: 8 max: 19
Platform ['Facebook', 'Instagram', 'LinkedIn', 'TikTok', 'X.com', 'YouTube']
Region ['Australia', 'Brazil', 'Canada', 'Germany', 'India', 'Japan', 'UK', 'USA']
Content_Category ['Customer Story', 'Educational', 'Entertainment', 'Event / Webinar', 'Product Promotion']
Post_Type ['Article', 'Carousel', 'Image', 'Live Stream', 'PDF', 'Text', 'Video']
Content_Type ['Organic', 'Sponsored']
Engagement_Level ['High', 'Low', 'Medium']
Click data coverage (not performance):
           records  non_null
Platform                    
Facebook       299       299
Instagram      998         0
LinkedIn       525       304
TikTok        1257      1257
X.com         1201         0
YouTube       1320         0


In [4]:
# Definition checks only: candidate formulas are not validated business definitions.
print("Records where Engagement equals Likes + Shares + Comments:",
      df["Engagement"].eq(df["Likes"] + df["Shares"] + df["Comments"]).sum())
for label, observed, candidate in [
    ("Engagement_Rate vs Engagement / Views", df["Engagement_Rate"], df["Engagement"] / df["Views"]),
    ("Click_Through_Rate vs Clicks / Impressions", df["Click_Through_Rate"], df["Clicks"] / df["Impressions"])
]:
    difference = (observed - candidate).abs()
    print(label, "max absolute difference:", difference.max())
print("No source rows were modified or removed.")

Records where Engagement equals Likes + Shares + Comments: 0
Engagement_Rate vs Engagement / Views max absolute difference: 0.0009581404581550101
Click_Through_Rate vs Clicks / Impressions max absolute difference: 0.0008448275862068966
No source rows were modified or removed.
